In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

# BUNDLE_PATH = "C:/Users/avyaa/secure-connect-mxed-benchmarking.zip"  # forward slashes
# TOKEN
# KEYSPACE = "ecommerce"

# auth = PlainTextAuthProvider('token', TOKEN)
# cluster = Cluster(cloud={'secure_connect_bundle': BUNDLE_PATH}, auth_provider=auth)
# session = cluster.connect(KEYSPACE)

print("Connected:", KEYSPACE)

Connected: ecommerce


Products:   0%|                                                                                                                                                                       | 0/12000 [15:27<?, ?rows/s]


In [2]:
import uuid
from datetime import datetime
from decimal import Decimal

def U(s):
    try: return uuid.UUID(s)
    except: return uuid.uuid4()

def T(s):
    try: return datetime.fromisoformat(s.replace('T',' '))
    except: return datetime.now()

def D(s):
    try: return Decimal(str(s))
    except: return Decimal('0.00')

def I(s):
    try: return int(s)
    except: return 0

def B(s):
    return str(s).lower() in ('true','1','yes')

def S(s):
    return s if s and str(s).strip() else ""

In [3]:
def get_table_columns(session, keyspace, table):
    rows = session.execute(
        "SELECT column_name FROM system_schema.columns WHERE keyspace_name=%s AND table_name=%s",
        (keyspace, table)
    )
    return {r.column_name for r in rows}

In [4]:
import pandas as pd
from tqdm import tqdm
from cassandra.concurrent import execute_concurrent_with_args

def load_psv_dynamic(session, keyspace, table, csv_path, preferred_cols, type_map,
                     chunk_size=None, concurrency=None, progress_label=None):
    actual = get_table_columns(session, keyspace, table)
    df_iter = (pd.read_csv(csv_path, sep="|", dtype=str, keep_default_na=False)
               if not chunk_size else
               pd.read_csv(csv_path, sep="|", dtype=str, keep_default_na=False, chunksize=chunk_size))
    first_df = next(df_iter) if chunk_size else df_iter
    csv_cols = set(first_df.columns)
    chosen = [c for c in preferred_cols if c in actual and c in csv_cols]
    placeholders = ", ".join(["?"] * len(chosen))
    col_list = ", ".join(chosen)
    cql = f"INSERT INTO {table} ({col_list}) VALUES ({placeholders})"
    stmt = session.prepare(cql)
    print(f"[{table}] Using columns: {chosen}")

    def row_args(row):
        vals = []
        for c in chosen:
            caster = type_map.get(c, S)
            vals.append(caster(row[c]))
        return tuple(vals)

    loaded = errors = 0
    def proc_df(df):
        nonlocal loaded, errors
        if concurrency:
            args = [row_args(r) for _, r in df.iterrows()]
            res = execute_concurrent_with_args(session, stmt, args, concurrency=concurrency, raise_on_first_error=False)
            for ok, _ in res:
                if ok: loaded += 1
                else: errors += 1
        else:
            for _, r in df.iterrows():
                try:
                    session.execute(stmt, row_args(r))
                    loaded += 1
                except Exception:
                    errors += 1

    # process first chunk
    pbar_total = (sum(1 for _ in open(csv_path, encoding="utf-8", errors="ignore")) - 1) if chunk_size else len(first_df)
    pbar = tqdm(total=pbar_total, desc=progress_label or table, unit="rows")
    proc_df(first_df)
    pbar.update(len(first_df))

    # remaining chunks
    if chunk_size:
        for df in df_iter:
            proc_df(df)
            pbar.update(len(df))
    pbar.close()
    print(f"[{table}] Loaded: {loaded:,} | Errors: {errors:,}")

In [5]:
cat_cols_pref = ['id','name','name_lc','description','parent_id','is_active','created_at','updated_at']
cat_types = {
    'id': U, 'name': S, 'name_lc': S, 'description': S, 'parent_id': U,
    'is_active': B, 'created_at': T, 'updated_at': T
}
load_psv_dynamic(
    session, KEYSPACE, 'categories',
    "C:/Users/avyaa/Astra_dataset_psv/categories.psv",
    cat_cols_pref, cat_types,
    chunk_size=None, concurrency=None, progress_label="Categories"
)

[categories] Using columns: ['id', 'name', 'name_lc', 'parent_id', 'created_at', 'updated_at']


Categories: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:09<00:00,  5.06rows/s]

[categories] Loaded: 50 | Errors: 0


In [6]:
cust_cols_pref = ['id','first_name','last_name','email','email_lc','phone',
                  'address_line1','address_line2','city','state','country',
                  'postal_code','is_active','created_at','updated_at']
cust_types = {
    'id': U, 'first_name': S, 'last_name': S, 'email': S, 'email_lc': S, 'phone': S,
    'address_line1': S, 'address_line2': S, 'city': S, 'state': S, 'country': S,
    'postal_code': S, 'is_active': B, 'created_at': T, 'updated_at': T
}
load_psv_dynamic(
    session, KEYSPACE, 'customers',
    "C:/Users/avyaa/Astra_dataset_psv/customers.psv",
    cust_cols_pref, cust_types,
    chunk_size=50_000, concurrency=128, progress_label="Customers (conc=128)"
)

[customers] Using columns: ['id', 'first_name', 'last_name', 'email', 'email_lc', 'phone', 'address_line1', 'address_line2', 'city', 'state', 'country', 'postal_code', 'is_active', 'created_at', 'updated_at']


Customers (conc=128): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25000/25000 [00:41<00:00, 602.31rows/s]

[customers] Loaded: 25,000 | Errors: 0


In [10]:
# =========================
# FAST PRODUCTS LOADER
# =========================
import pandas as pd
from tqdm import tqdm
from cassandra.concurrent import execute_concurrent_with_args

prod_path = "C:/Users/avyaa/Astra_dataset_psv/products.psv"
CHUNK = 50_000
CONCURRENCY = 192  # tune: 128–256

stmt_prod = session.prepare("""
INSERT INTO products (id, sku, sku_lc, name, name_lc, description, category_id,
                      brand, brand_lc, price, currency, stock, popularity,
                      attributes, is_active, created_at, updated_at)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

total = sum(1 for _ in open(prod_path, encoding="utf-8", errors="ignore")) - 1
pbar = tqdm(total=total, desc=f"Products (concurrency={CONCURRENCY})", unit="rows")
ok = err = 0

for df in pd.read_csv(prod_path, sep="|", dtype=str, keep_default_na=False, chunksize=CHUNK):
    args = [(
        U(r['id']), S(r['sku']), S(r['sku_lc']), S(r['name']), S(r['name_lc']),
        S(r['description']), U(r['category_id']), S(r['brand']), S(r['brand_lc']),
        D(r['price']), S(r['currency']), I(r['stock']), I(r['popularity']),
        S(r['attributes']), B(r.get('is_active','true')), T(r['created_at']), T(r['updated_at'])
    ) for _, r in df.iterrows()]

    results = execute_concurrent_with_args(session, stmt_prod, args, concurrency=CONCURRENCY, raise_on_first_error=False)
    for success, _ in results:
        ok += 1 if success else 0
        err += 0 if success else 1
        pbar.update(1)

pbar.close()
print(f"Products loaded: {ok:,} | Errors: {err:,}")



Products (concurrency=192):   0%|                                                                                                                                                     | 0/12000 [00:00<?, ?rows/s]

Products (concurrency=192): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12000/12000 [00:12<00:00, 956.54rows/s]

Products loaded: 12,000 | Errors: 0


In [11]:
# =========================
# FAST ORDERS LOADER
# =========================
orders_path = "C:/Users/avyaa/Astra_dataset_psv/orders.psv"
CHUNK = 50_000
CONCURRENCY = 128

stmt_orders = session.prepare("""
INSERT INTO orders (id, customer_id, order_date, status, payment_status, currency,
                    subtotal_amount, tax_amount, shipping_fee, discount_amount,
                    total_amount, coupon_code, shipping_address, billing_address,
                    created_at, updated_at)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

total = sum(1 for _ in open(orders_path, encoding="utf-8", errors="ignore")) - 1
pbar = tqdm(total=total, desc=f"Orders (concurrency={CONCURRENCY})", unit="rows")
ok = err = 0

for df in pd.read_csv(orders_path, sep="|", dtype=str, keep_default_na=False, chunksize=CHUNK):
    args = [(
        U(r['id']), U(r['customer_id']), T(r['order_date']), S(r['status']), S(r['payment_status']),
        S(r['currency']), D(r['subtotal_amount']), D(r['tax_amount']), D(r['shipping_fee']),
        D(r['discount_amount']), D(r['total_amount']), S(r['coupon_code']),
        S(r['shipping_address']), S(r['billing_address']), T(r['created_at']), T(r['updated_at'])
    ) for _, r in df.iterrows()]

    results = execute_concurrent_with_args(session, stmt_orders, args, concurrency=CONCURRENCY, raise_on_first_error=False)
    for success, _ in results:
        ok += 1 if success else 0
        err += 0 if success else 1
        pbar.update(1)

pbar.close()
print(f"Orders loaded: {ok:,} | Errors: {err:,}")

Orders (concurrency=128): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250000/250000 [06:53<00:00, 603.98rows/s]

Orders loaded: 250,000 | Errors: 0


In [12]:
# =========================
# FAST ORDER_ITEMS LOADER
# =========================
oi_path = "C:/Users/avyaa/Astra_dataset_psv/order_items.psv"
CHUNK = 75_000
CONCURRENCY = 160  # tune: 128–192

stmt_oi = session.prepare("""
INSERT INTO order_items (id, order_id, product_id, quantity, unit_price,
                         discount_amount, total_price, created_at)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")

total = sum(1 for _ in open(oi_path, encoding="utf-8", errors="ignore")) - 1
pbar = tqdm(total=total, desc=f"OrderItems (concurrency={CONCURRENCY})", unit="rows")
ok = err = 0

for df in pd.read_csv(oi_path, sep="|", dtype=str, keep_default_na=False, chunksize=CHUNK):
    args = [(
        U(r['id']), U(r['order_id']), U(r['product_id']), I(r['quantity']),
        D(r['unit_price']), D(r['discount_amount']), D(r['total_price']), T(r['created_at'])
    ) for _, r in df.iterrows()]

    results = execute_concurrent_with_args(session, stmt_oi, args, concurrency=CONCURRENCY, raise_on_first_error=False)
    for success, _ in results:
        ok += 1 if success else 0
        err += 0 if success else 1
        pbar.update(1)

pbar.close()
print(f"Order items loaded: {ok:,} | Errors: {err:,}")

OrderItems (concurrency=160): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500000/500000 [12:03<00:00, 691.07rows/s]

Order items loaded: 500,000 | Errors: 0


In [13]:
for t in ['products','orders','order_items']:
    try:
        c = session.execute(f"SELECT COUNT(*) FROM {t}", timeout=300).one().count
        print(f"{t}: {c:,}")
    except Exception:
        print(f"{t}: count timed out (data likely loaded)")

products: 12,000
orders: 250,000
order_items: 500,000
